# 03 — Driving Safety Analysis with Chain-of-Thought

**What this notebook demonstrates:**
- Chain-of-thought (CoT) reasoning with `reasoning=True`
- Dashcam video analysis for autonomous driving safety
- The `<think>...</think>` reasoning format

**Key concept:** When `reasoning=True`, Cosmos-Reason2 first produces a
`<think>` block with step-by-step reasoning, then outputs its final answer.
This mimics how a safety system would decompose a driving scene.

---

In [ ]:
import os
os.environ['QWEN_VL_VIDEO_READER'] = 'torchcodec'

import sys, os, time
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
project_root = os.path.abspath('..')
sample_video = os.path.join(project_root, 'sample.mp4')
assert os.path.exists(sample_video), f"Missing: {sample_video}"

## Load Model with Chain-of-Thought Enabled

Setting `reasoning=True` appends the reasoning format prompt automatically.
The model will produce `<think>reasoning steps</think>` before its answer.

In [ ]:
from strands import Agent
from strands_cosmos import CosmosVisionModel

t0 = time.time()
model = CosmosVisionModel(
    model_id="nvidia/Cosmos-Reason2-2B",
    reasoning=True,          # Enable chain-of-thought
    fps=4,                   # 4 frames/sec from dashcam
    params={"max_tokens": 2048, "temperature": 0.6},
)
agent = Agent(model=model)
print(f"✅ CoT model ready in {time.time() - t0:.1f}s")

## Driving Safety Analysis

The driving task prompt from Cosmos-Reason2:
> *"The video depicts the observation from the vehicle's camera. You need to think
> step by step and identify the objects in the scene that are critical for safe navigation."*

Let's use a custom safety-focused prompt:

In [ ]:
t0 = time.time()
result = agent(
    f"<video>{sample_video}</video> "
    "Analyze this dashcam footage for safety hazards. "
    "Identify: 1) moving objects, 2) potential collision risks, 3) recommended driver actions."
)
print(f"\n⏱  Total time: {time.time() - t0:.1f}s")

## Temporal Event Detection

In [ ]:
t0 = time.time()
result = agent(
    f"<video>{sample_video}</video> "
    "Describe the notable events in this video in chronological order."
)
print(f"\n⏱  Total time: {time.time() - t0:.1f}s")

---
## Summary

| Metric | Value |
|--------|-------|
| Model | Cosmos-Reason2-2B (Vision + CoT) |
| Mode | Video → Think → Answer |
| Use case | Autonomous driving safety |
| Typical latency | 3-8s (longer due to reasoning) |

**Next:** [04_embodied_reasoning.ipynb](04_embodied_reasoning.ipynb) — robot action prediction from images.